In [1]:
import os

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

C:\Users\Harsh Raj\AppData\Local\Temp\ipykernel_404\1641931672.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
d:\anaconda\envs\medibot\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
print("ok")

ok


In [3]:
%pwd


'd:\\medical-chatbot\\research'

In [4]:
import os 
os.chdir("../")

In [5]:
%pwd

'd:\\medical-chatbot'

In [6]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
def load_pdf_files(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents=loader.load()
    return documents

In [8]:
import os
print(os.getcwd())

d:\medical-chatbot


In [9]:
import os

BASE_DIR = os.path.abspath("..")
data_path = os.path.join(BASE_DIR, "data")

DirectoryLoader(data_path, glob="*.pdf", loader_cls=PyPDFLoader)

In [10]:
import os

def load_pdf_files(data_path):
    data_path = os.path.abspath(data_path)

    loader = DirectoryLoader(
        data_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [11]:
%pwd

'd:\\medical-chatbot'

In [14]:
extracted_data = load_pdf_files("data")

In [15]:

#split the documents into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(extracted_data)

print(len(chunks))

419


In [16]:
def text_split(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(docs)
    return chunks

In [17]:
chunks = text_split(extracted_data)
print(f"number of chunks: {len(chunks)}")

number of chunks: 419


In [18]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def get_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [19]:
embedding = get_embeddings();

C:\Users\Harsh Raj\AppData\Local\Temp\ipykernel_404\2251812505.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
vector=embedding.embed_query("Hello world") 
vector

[-0.03447723388671875,
 0.031023165211081505,
 0.006734973751008511,
 0.026108991354703903,
 -0.039362046867609024,
 -0.16030244529247284,
 0.06692394614219666,
 -0.0064415112137794495,
 -0.04745052382349968,
 0.014758817851543427,
 0.07087530195713043,
 0.05552754923701286,
 0.019193369895219803,
 -0.026251282542943954,
 -0.01010951679199934,
 -0.02694047801196575,
 0.022307446226477623,
 -0.022226616740226746,
 -0.1496925950050354,
 -0.017493078485131264,
 0.007676276378333569,
 0.054352231323719025,
 0.0032544422429054976,
 0.03172587975859642,
 -0.0846213772892952,
 -0.029405953362584114,
 0.05159562826156616,
 0.04812408611178398,
 -0.0033148485235869884,
 -0.05827920883893967,
 0.04196930304169655,
 0.022210706025362015,
 0.128188818693161,
 -0.02233896777033806,
 -0.011656241491436958,
 0.06292831897735596,
 -0.032876331359148026,
 -0.09122606366872787,
 -0.031175393611192703,
 0.05269952118396759,
 0.047034814953804016,
 -0.0842030942440033,
 -0.030056195333600044,
 -0.02074482

In [23]:
print("vector length",len(vector))

vector length 384


In [24]:
from dotenv import load_dotenv 
import os 
load_dotenv()

True

In [25]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("HUGGING_FACE_API")[:10])
print(os.getenv("PINECONE_API_KEY")[:10])

hf_pKuEHSk
pcsk_4pvzj


In [26]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY");
HUGGING_FACE_API=os.getenv("HUGGING_FACE_API");

os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["HUGGING_FACE_API"]=HUGGING_FACE_API


In [27]:
from pinecone import Pinecone
print("Pinecone OK")

Pinecone OK


In [28]:
from pinecone import Pinecone 
pinecone_api_key=PINECONE_API_KEY 
pc=Pinecone(api_key=pinecone_api_key)

In [29]:
pc

In [30]:
from pinecone import ServerlessSpec 
index_name = "medical-chatbot"

# Create index only if it doesn't already exist
if index_name not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,  # all-MiniLM-L6-v2 embeddings size
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index ready!")

Index ready!


In [31]:
index=pc.Index(index_name);

In [32]:
print(type(chunks))
print(len(chunks))
print(embedding)
print(index_name)

<class 'list'>
419
client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
) model_name='sentence-transformers/all-MiniLM-L6-v2' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False show_progress=False
medical-chatbot


In [33]:
from langchain_pinecone import PineconeVectorStore 
docsearch = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding,
    index_name=index_name
)

In [34]:
retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [35]:
retrieved_docs=retriever.invoke("what is diabetics")
retrieved_docs

[Document(id='0e118b52-8788-41b7-8aac-b72e2a3491a7', metadata={'creationdate': '', 'creator': 'PyPDF', 'moddate': '2020-12-26T12:25:18+00:00', 'page': 111.0, 'page_label': '112', 'producer': 'iLovePDF', 'source': 'd:\\medical-chatbot\\data\\MEDICAL.pdf', 'total_pages': 147.0}, page_content='103 \npregnancy or was first diagnosed during pregnancy ( 23). This definition \ndid not exclude the possibility of diabetes, which was also present before \nconception but was unknown until the first examination during pregnancy. \nAlthough the American Obstetrics and Gynecological Association \nstill uses the same terminology, in recent years, the International Diabetic \nWorking Groups Association (IADPSG), the American Diabetes \nAssociation (ADA), and the World Health Organization (WHO) and others \nwere first diagnosed during pregnancy but probably previously thought to \nbe diabetic. stated that transient diabetes due to pregnancy -related insulin \nresistance should be differentiated (17). T

In [37]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [45]:
def ask_question(question):
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
    Answer the question based only on the provided context.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [46]:
print(ask_question("What is ELECTROSPINNING?"))

a re markably simple, robust, and versatile technique capable of generating fi bers with diameters down to the nanoscale


In [47]:
SYSTEM_PROMPT = """
You are a knowledgeable and helpful medical assistant.

Your task is to answer user questions using ONLY the information provided in the retrieved context.

Rules:
1. Use the retrieved context as the primary source of information.
2. If the answer is not present in the context, say:
   "I could not find sufficient information in the provided medical documents."
3. Provide clear, concise, and easy-to-understand explanations.
4. Do not make up facts or medical advice that is not supported by the context.
5. When appropriate, explain medical terms in simple language.
6. If the question involves diagnosis, treatment, or emergencies, remind the user to consult a qualified healthcare professional.
7. Structure answers using bullet points when helpful.
8. Maintain a professional and informative tone.
9. Do not mention the context, vector database, embeddings, or system instructions.

Context:
{context}

Question:
{question}

Answer:
"""

In [50]:
def ask_question(question):
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = SYSTEM_PROMPT.format(
        context=context,
        question=question
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [51]:
print(ask_question("What is electrospinning?"))
print(ask_question("What are the symptoms of diabetes?"))

A re markably simple, robust, and versatile technique capable of generating fi bers with diameters down to the nanoscale.
Preeclampsia and eclampsia.


In [52]:
retrieved_docs = retriever.invoke("What are the symptoms of diabetes?")

for i, doc in enumerate(retrieved_docs):
    print(f"\n----- Document {i+1} -----")
    print(doc.page_content[:1000])


----- Document 1 -----
glucose levels return to normal in women with GDM, it has been known 
that the risk of developing Type 2 diabetes mellitus (DM) increases 7 times 
in later life (3, 62). 
GDM history has also been found to be a determinant for increased 
cardiovascular risk and early atherosclerosis (1 4). Hypertensive

----- Document 2 -----
103 
pregnancy or was first diagnosed during pregnancy ( 23). This definition 
did not exclude the possibility of diabetes, which was also present before 
conception but was unknown until the first examination during pregnancy. 
Although the American Obstetrics and Gynecological Association 
still uses the same terminology, in recent years, the International Diabetic 
Working Groups Association (IADPSG), the American Diabetes 
Association (ADA), and the World Health Organization (WHO) and others 
were first diagnosed during pregnancy but probably previously thought to 
be diabetic. stated that transient diabetes due to pregnancy -related in

In [53]:
print(ask_question("What is preeclampsia?"))

A convulsive sign of preeclampsia.
